 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import pandas as pd
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2


In [ ]:

citenum_url_link_re = re.compile(r'\[(?P<orig>\d+)\]\((?P<url>https?://[^\)]+)\)')
sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)')
#citenum_plain_re = re.compile(r'\[(\d+)\]')
citenum_plain_re = re.compile(r'\[(?P<num>\d+)\]')

def find_source_dup_urls(source_matches):
    
    url_to_match = defaultdict(list)
    for match in source_matches:
        url_to_match[match.group('url')].append(match)
    
    if len(url_to_match) == len(source_matches):
        #print('no reumbering needed')
        return source_matches, None # no renumbering needed
    
    #print('Renumbering source links')

    # create new citenums if there are dups
    new_cite_num = 1
    lut = []
    source_matches_renumbered = []
    for url, matches in url_to_match.items():
        if (nDups := len(matches)) > 1:
            nums = [m['num'] for m in matches]
            print(f'URL has {nDups} dups: {nums=}, {url=}')

        for ix, match in enumerate(matches):
            lut.append({'orig_num': match['num'], 'new_num': str(new_cite_num), 'url': url})
            #if ix == 0:
                #ic('appending', match)
                # this removes a match but leaves the numbering the same.
                # instead of reducing the number of source matches, I just need to renumber them
                # source_matches_renumbered.append(match)  # cite number of 1st URL match
        
        new_cite_num += 1
    
    ic(len(source_matches_renumbered),len(source_matches))
    smr_tmp = source_matches_renumbered[-10:]
    ic(smr_tmp)
    
    return source_matches_renumbered, pd.DataFrame(lut).set_index(['orig_num'])

def split_body_source(perplexity_file: pl.Path, renumber_dup_cites: bool = False):
    """Replace links in standard Perplexity (saved clipboard) output with links 
    to Zotero items or Obsidian lit notes. """    
    
    content = perplexity_file.read_text(encoding='utf-8')
    section_parts = content.split("\nCitations:\n", 1)
    if len(section_parts) < 2:
        print("Missing citations")
        body, citations = section_parts, ""
    else:
        body, citations = section_parts
    
    source_matches = list(re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M))
    
    source_matches_renumbered, old_to_new_num = find_source_dup_urls(source_matches)
    # good: ic(old_to_new_num) # good, works
    ic(source_matches_renumbered[-10:]) # BAD!
    if renumber_dup_cites:
        print("Renumbering body and source links:")
        source_matches = source_matches_renumbered  # means that body must be renumbered too
        ic(old_to_new_num, source_matches)
    
    
    return body, source_matches, old_to_new_num

def relink_chunks(body, source_matches, old_to_new_citenum=None) -> None:

    def make_relinks_from_source(cite_num: str, doc_url: str) -> str:
        """Returns what a relinked citation would look like if present in the body,
        given a source part citation number and url.  Also appends to the global list, 
        relinked_sources, a relinked source part link.  Expects the global set, body_cite_nums."""
        
        numbered_link = f"[{cite_num}]({doc_url})"
        if zotero_item := relinker.find_zotero_item_via_url(doc_url):
            body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
            relinked_sources.append(f'({numbered_link}) **{body_link}**')
        else:
            body_link = f"=={numbered_link}==" # mark it as "not in zotero"
            source_line = f'({numbered_link}) {doc_url}'
            source_line = f'=={source_line} ==' if cite_num in body_cite_nums else source_line
            relinked_sources.append(source_line)
            
        return body_link
    
    display(old_to_new_citenum)

    if old_to_new_citenum is not None:
        print('Renumbering body.  If here, sources s/b already renumbered')
        ic(len(source_matches))
        ic('in relink',old_to_new_citenum)
        old_to_new = old_to_new_citenum.new_num.to_dict()
        ic(old_to_new)        
        def replace_body_citenum(match):
            """Replace a cite number in the body with a new cite number.  This is used to renumber
            the body if duplicate cite numbers are found in the source list."""
            return f'[{old_to_new[match.group("num")]}]'

        body = re.sub(citenum_plain_re, replace_body_citenum, body)

    relinker = lpz.ZoteroLinkConverter()
    relinked_sources = []
    body_cite_nums = set(re.findall(citenum_plain_re, body))
    new_num_to_url.setindex('new_num').to_dict()
    source_num_to_link = {num: make_relinks_from_source(num, url)
                          num, url for m in new_num_to_url.items() }
    # source_num_to_link = {m.group('num'): make_relinks_from_source(m.group('num'), m.group('url'))
    #                       for m in source_matches }
    body_relinked = re.sub(citenum_plain_re, 
                           lambda m: f' {source_num_to_link.get(m.group(1))}', body)
    sources_relinked = "\n".join(relinked_sources)
    
    return body_relinked, sources_relinked

def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path, renumber_dup_cites=False) -> None:
    body, source_matches, old_to_new_citenum = split_body_source(perplexity_file, renumber_dup_cites)
    body_relinked, sources_relinked = relink_chunks(body, source_matches, old_to_new_citenum)
    ic(sources_relinked)
    relinked_file.write_text(f'# Response\n{body_relinked}\n# Citations\n{sources_relinked}', encoding='utf-8')
    


In [36]:
remove_dup_cites_example = True
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_new_cites_perplexity_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')

relink_perplexity_export(perplexity_dialog_file, output_file, remove_dup_cites_example)
print('Done.')


ic| len(source_matches_renumbered): 0, len(source_matches): 81
ic| smr_tmp: []
ic| source_matches_renumbered[-10:]: []
ic| old_to_new_num:          new_num                                                url
                    orig_num                                                           
                    1              1  https://en.wikipedia.org/wiki/Right-wing_populism
                    2              2  https://www.politico.eu/article/mapped-europe-...
                    3              3  https://www.npr.org/2024/06/09/nx-s1-4997712/f...
                    4              4  https://globalaffairs.org/commentary-and-analy...
                    54             4  https://globalaffairs.org/commentary-and-analy...
                    ...          ...                                                ...
                    77            76                    https://rioonwatch.org/?p=72542
                    78            77  https://www.populismstudies.org/chega-emerges-...
 

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')
URL has 2 dups: nums=['4', '54'], url='https://globalaffairs.org/commentary-and-analysis/blogs/brazils-systemic-mistrust-elections-and-democracy'
Renumbering body and source links:


for-portugal-a...
                    81            80  https://www.statista.com/topics/9604/far-right...
                    
                    [81 rows x 2 columns]
    source_matches: []


,new_num,url
orig_num,,
1,1,https://en.wikipedia.org/wiki/Right-wing_populism
2,2,https://www.politico.eu/article/mapped-europe-...
3,3,https://www.npr.org/2024/06/09/nx-s1-4997712/f...
4,4,https://globalaffairs.org/commentary-and-analy...
54,4,https://globalaffairs.org/commentary-and-analy...
...,...,...
77,76,https://rioonwatch.org/?p=72542
78,77,https://www.populismstudies.org/chega-emerges-...
79,78,https://www.american.edu/sis/centers/transatla...


ic| len(source_matches): 0
ic| 'in relink': 'in relink'
    old_to_new_citenum:          new_num                                                url
                        orig_num                                                           
                        1              1  https://en.wikipedia.org/wiki/Right-wing_populism
                        2              2  https://www.politico.eu/article/mapped-europe-...
                        3              3  https://www.npr.org/2024/06/09/nx-s1-4997712/f...
                        4              4  https://globalaffairs.org/commentary-and-analy...
                        54             4  https://globalaffairs.org/commentary-and-analy...
                        ...          ...                                                ...
                        77            76                    https://rioonwatch.org/?p=72542
                        78            77  https://www.populismstudies.org/chega-emerges-...
                        

Renumbering body.  If here, sources s/b already renumbered


'1',
                 '10': '10',
                 '11': '11',
                 '12': '12',
                 '13': '13',
                 '14': '14',
                 '15': '15',
                 '16': '16',
                 '17': '17',
                 '18': '18',
                 '19': '19',
                 '2': '2',
                 '20': '20',
                 '21': '21',
                 '22': '22',
                 '23': '23',
                 '24': '24',
                 '25': '25',
                 '26': '26',
                 '27': '27',
                 '28': '28',
                 '29': '29',
                 '3': '3',
                 '30': '30',
                 '31': '31',
                 '32': '32',
                 '33': '33',
                 '34': '34',
                 '35': '35',
                 '36': '36',
                 '37': '37',
                 '38': '38',
                 '39': '39',
                 '4': '4',
                 '40': '40',
               

Reading from cache.


ic| sources_relinked: ''


Done.


### Test merging

In [ ]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

chat_files = list(datdir.glob('*.md'))
chat_files

merged_output_file = tmpdir / 'tmp_stock_perplexy_merged.md'

In [ ]:
chat_files

In [ ]:
this is broken: old_to_new_citenums are accumulated correctly, and are kind of required twice

renumber_dup_cites = True

# get the sources from all docs to be merged
all_bodies, all_old_to_new_citenums, all_source_matches = [], [], []
for chat_file in chat_files:
    body, old_to_new_citenum, source_matches = split_body_source(chat_file)
    all_bodies.append(body)
    all_source_matches.append(source_matches)
    all_old_to_new_citenums.append(old_to_new_citenum)

# find all the doc citations for each unique URL
url_to_match = defaultdict(list)
print(url_to_match)
for docIx, old_to_new_citenum in enumerate(all_old_to_new_citenums):
    for url, num in old_to_new_citenum.items():
        url_to_match[url].append(dict(orig_num=num, docIx=docIx))

# create new citenums for a combined document with a combined sources section
new_cite_num = 1
lut = []
for url, nums in url_to_match.items():
    for info in nums:
        lut.append({'url': url, 'new_cite_num': str(new_cite_num)} | info)
    new_cite_num += 1

lut = pd.DataFrame(lut).set_index(['docIx', 'orig_num'])
all_new_cite_nums = lut.new_cite_num.unique()


# make single body with cite numbers replaced by combined cite numbers

body_cites_not_in_sources = []
def replace_link_num(m):
    orig_citenum = m.group('orig')
    try:
        num = old_to_new_citenum[orig_citenum]
    except:
        chat_file=chat_files[docIx]
        print(f"Missing source for {orig_citenum} in {chat_file}")
        info = dict(orig_cite_num=orig_citenum, chat_file=chat_file)
        body_cites_not_in_sources.append(info)
        num = orig_citenum

    return f"[{num}]({m.group('url')})"



concat_bodies = ""
concat_sources = ""
for docIx, body in enumerate(all_bodies):
    # relink body with old citenums
    source_matches = all_source_matches[docIx]
    url_to_source_nums = all_url_to_source_nums[docIx]
    body_relinked, sources_relinked = relink_chunks(body, source_matches, old_to_new_citenum)
    
    old_to_new_citenum =lut.loc[docIx].new_cite_num.to_dict()
    body_re_relinked = re.sub(citenum_url_link_re, replace_link_num, body_relinked)
    concat_bodies += f'# {chat_files[docIx].name}\n{body_re_relinked}\n'
    sources_re_relinked = re.sub(citenum_url_link_re, replace_link_num, sources_relinked)
    concat_sources += f'{sources_re_relinked}\n'

In [ ]:
import numpy as np
len(concat_sources.split('\n')), len(np.unique(concat_sources.split('\n')))

concat_sources_unique = list(set(concat_sources.split('\n')))
#sorted_strings = sorted(concat_sources_unique, key=lambda x: int(re.search(r'\((\d+)\]', x).group(1)))
#sorted_strings

# Function to extract the number inside [num]
def extract_number(s):
    match = re.search(r'\[(\d+)\]', s)
    return int(match.group(1)) if match else None  # Handle cases without [num]

# Sort the list using the extracted number as key
merged_sources = "\n".join(sorted(concat_sources_unique, key=extract_number))

ic(merged_output_file)
merged_output_file.write_text(f'# Responses\n{concat_bodies}\n# Citations\n{merged_sources}', encoding='utf-8')
print('Done.')

In [ ]:
lut